In [ ]:
# see link
# https://github.com/Fraud-Detection-Handbook

package ='25-sklearn.neural'
name='MLP'
tuningAndParameters='02-After tuning'

hyperparametersFound = {}
scalerFound='StandardScaler'


print('done')

In [ ]:
import sys
import os
from importlib import reload
fpath = os.path.join('..//scripts')
sys.path.append(fpath)

import warnings
warnings.filterwarnings('ignore')

#loading internal scripts
import datamanagement as dm
reload(dm)

import result as resultMd
reload(resultMd)

import graph as gf
reload(gf)

import scaler as scaler
reload(scaler)

print('done')

In [ ]:
dfLearning, dfValidation =dm.getDataLearningAndValidation()

dfLearning.head()

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x_train, x_test, y_train, y_test = train_test_split(dfLearning[predictors], dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

# No scaling no result

In [ ]:
#%%script false

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score


modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(3,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)

modelClf.fit(x_train, y_train)


predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)

f1Learning =f1_score(y_train, predsTrain)
f1Test=f1_score(y_test, predsTest)
dm.show_confusion_matrix(y_train, predsTrain,'Confusion matrix learning data')
print(f"f1 train {f1Learning:.3f}")
dm.show_confusion_matrix(y_test, predsTest,'Confusion matrix test data')
print(f"f1 test {f1Test:.3f}")
resultMd.update_learning_test_result(package, name, tuningAndParameters, f1Learning,f1Test)

# Scaling choice

In [ ]:
#%script false

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import NearMiss
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()
scalingData=[]

scalers = scaler.getScalers()
for key in scalers:
    print(key)
    x1, y1 = dfLearning[predictors], dfLearning[target]
    sc=scalers.get(key)
    x2 = sc.fit_transform(x1)

    TEST_SIZE = 0.20 # test size using_train_test_split
    RANDOM_STATE = 0


    x_train0, x_test, y_train0, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)


    
    modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(3,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)

    modelClf.fit(x_train0, y_train0)
    predsTrain = modelClf.predict(x_train0)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train0, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    print('-----------------------')
    subScalingData = [train_f1,test_f1]
    scalingData.append(subScalingData)

print(scalingData)
import matplotlib.pyplot as plt

fig = plt.figure(figsize =(10, 7))
ax = fig.add_subplot(111)
bp = ax.boxplot(scalingData, patch_artist = True,
                notch ='True', vert = 0)
ax.set_xticklabels(['StandardScaler','MinMaxScaler','RobustScaler','MaxAbsScaler'])
plt.title("Scaling choice")
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
plt.show()

# Hyperparameter choice

## hidden layer

In [ ]:

def plt_train_test(range, tabf1Train,trainLabel="f1 Train",tabf1Test=[], testLabel="f1 test"):
    fig = plt.figure(figsize=(8,6))
    ax1 = fig.add_subplot()

    ax1.set_ylabel(trainLabel)
    ax1.plot(range, tabf1Train, color = 'red', label = trainLabel)
    ax1.legend(loc = 'upper left')

    if(len(tabf1Test)==len(tabf1Train)):
        ax2 = ax1.twinx()
        ax2.set_ylabel(testLabel)
        ax2.plot(range, tabf1Test, color = 'blue', label = testLabel)
        ax2.legend(loc = 'upper right')

    fig.autofmt_xdate()
    plt.show()


x1, y1 = dfLearning[predictors], dfLearning[target]
sc=scalers.get(scalerFound)
x2 = sc.fit_transform(x1)
TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0


x_train0, x_test, y_train0, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)


ranges = []
f1Test = []
f1Train = []
for hidden_layer in range(1, 21, 1):
    modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(hidden_layer,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)
    print("--- hidden_layer --", hidden_layer)
    modelClf.fit(x_train0, y_train0)
    predsTrain = modelClf.predict(x_train0)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train0, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    print('-----------------------')
    ranges.append(hidden_layer)
    f1Train.append(train_f1)
    f1Test.append(test_f1)

plt_train_test(ranges, f1Train,"f1 train",f1Test,"f1 test")

In [ ]:
plt_train_test(ranges, f1Train,"f1 train",f1Test,"f1 test")

In [ ]:
print('one hidden layer')
hidden_layer=7
modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(hidden_layer,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)
print("--- hidden_layer --", hidden_layer)
modelClf.fit(x_train0, y_train0)
predsTrain = modelClf.predict(x_train0)
predsTest = modelClf.predict(x_test)

train_f1=f1_score(y_train0, predsTrain)
print("f1 train {:.4f}".format(train_f1))
    
test_f1=f1_score(y_test, predsTest)
print("f1 test  {:.4f}".format(test_f1))


## Two hiden layers

In [ ]:

def plt_train_test(range, tabf1Train,trainLabel="f1 Train",tabf1Test=[], testLabel="f1 test"):
    fig = plt.figure(figsize=(8,6))
    ax1 = fig.add_subplot()

    ax1.set_ylabel(trainLabel)
    ax1.plot(range, tabf1Train, color = 'red', label = trainLabel)
    ax1.legend(loc = 'upper left')

    if(len(tabf1Test)==len(tabf1Train)):
        ax2 = ax1.twinx()
        ax2.set_ylabel(testLabel)
        ax2.plot(range, tabf1Test, color = 'blue', label = testLabel)
        ax2.legend(loc = 'upper right')

    fig.autofmt_xdate()
    plt.show()


x1, y1 = dfLearning[predictors], dfLearning[target]
sc=scalers.get(scalerFound)
x2 = sc.fit_transform(x1)
TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0


x_train0, x_test, y_train0, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)


ranges = []
f1Test = []
f1Train = []
for hidden_layer in range(1, 21, 1):
    modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(2*hidden_layer,hidden_layer,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)
    print("--- hidden_layer --", hidden_layer)
    modelClf.fit(x_train0, y_train0)
    predsTrain = modelClf.predict(x_train0)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train0, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    print('-----------------------')
    ranges.append(hidden_layer)
    f1Train.append(train_f1)
    f1Test.append(test_f1)

plt_train_test(ranges, f1Train,"f1 train",f1Test,"f1 test")

In [ ]:
plt_train_test(ranges, f1Train,"f1 train",f1Test,"f1 test")

In [ ]:
print('two hidden layers')
hidden_layer=8
modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(2*hidden_layer,hidden_layer,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)
print("--- hidden_layer --", hidden_layer)
modelClf.fit(x_train0, y_train0)
predsTrain = modelClf.predict(x_train0)
predsTest = modelClf.predict(x_test)

train_f1=f1_score(y_train0, predsTrain)
print("f1 train {:.4f}".format(train_f1))
    
test_f1=f1_score(y_test, predsTest)
print("f1 test  {:.4f}".format(test_f1))

## Three hiden layers

In [ ]:

def plt_train_test(range, tabf1Train,trainLabel="f1 Train",tabf1Test=[], testLabel="f1 test"):
    fig = plt.figure(figsize=(8,6))
    ax1 = fig.add_subplot()

    ax1.set_ylabel(trainLabel)
    ax1.plot(range, tabf1Train, color = 'red', label = trainLabel)
    ax1.legend(loc = 'upper left')

    if(len(tabf1Test)==len(tabf1Train)):
        ax2 = ax1.twinx()
        ax2.set_ylabel(testLabel)
        ax2.plot(range, tabf1Test, color = 'blue', label = testLabel)
        ax2.legend(loc = 'upper right')

    fig.autofmt_xdate()
    plt.show()


x1, y1 = dfLearning[predictors], dfLearning[target]
sc=scalers.get(scalerFound)
x2 = sc.fit_transform(x1)
TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0


x_train0, x_test, y_train0, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)


ranges = []
f1Test = []
f1Train = []
for hidden_layer in range(1, 21, 1):
    modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(3*hidden_layer,2*hidden_layer,hidden_layer,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)
    print("--- hidden_layer --", hidden_layer)
    modelClf.fit(x_train0, y_train0)
    predsTrain = modelClf.predict(x_train0)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train0, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    print('-----------------------')
    ranges.append(hidden_layer)
    f1Train.append(train_f1)
    f1Test.append(test_f1)

plt_train_test(ranges, f1Train,"f1 train",f1Test,"f1 test")

In [ ]:
plt_train_test(ranges, f1Train,"f1 train",f1Test,"f1 test")

In [ ]:
print('three hidden layers')
hidden_layer=3
modelClf = MLPClassifier(solver='sgd',
                        hidden_layer_sizes=(3*hidden_layer,2*hidden_layer,hidden_layer,),
                        activation='relu',
                        max_iter=1_000,
                        verbose=0, 
                        random_state=42,
                        learning_rate_init=0.05)
print("--- hidden_layer --", hidden_layer)
modelClf.fit(x_train0, y_train0)
predsTrain = modelClf.predict(x_train0)
predsTest = modelClf.predict(x_test)

train_f1=f1_score(y_train0, predsTrain)
print("f1 train {:.4f}".format(train_f1))
    
test_f1=f1_score(y_test, predsTest)
print("f1 test  {:.4f}".format(test_f1))
print('-----------------------')

# Summary

In [ ]:
print('Summary')
print(f"{package} {name} {tuningAndParameters}") 
print(f"hyperparameters {hyperparametersFound}") 
print(f"scaler {scalerFound}") 
print('-----------------------------')

print(f"learning duration {learningDurationInS:.2f} s")
print('-----------------------------')
print(f"f1 train      {f1Learning:.3f}")
print(f"f1 test       {f1Test:.3f}")
print(f"f1 validation {f1Validation:.3f}")